# Download the Bullinger (ENHG) train + test data

Fetches the [Bullinger HTR dataset](https://github.com/pstroe/bullinger-htr) — 16th-century
correspondence of Heinrich Bullinger — as pre-cropped line images (`.png` + `.txt` pairs).
Each archive unzips to `<range>/de` (Early New High German) and `<range>/la` (Latin). The model
in this project is German-adapted, so by default we keep only the **`de/`** subset.

## Archive layout (important)
The data is stored as **Git-LFS zip archives** served from the GitHub media endpoint, one set
per document-ID range. There are two shapes:

- **`test/`** — plain single `.zip` files → extract directly.
- **`train/`** — **split (multi-volume) zips**, max 2 GB per volume. Each set is
  `…-out.z01, …-out.z02, …, …-out.zip`. You must download **all** volume parts, then unify them
  with `zip -F` before extracting (per the dataset README):

  ```
  zip -F 0-2000-out.zip --out single-0-2000.zip
  ```

  Volume counts: `0-2000`→6+zip (~14 GB), `2001-4000`→4+zip, `4001-6000`→2+zip,
  `6001-8000`→1+zip, `8001-10000`→2+zip, `12001-14000`→3+zip. **Full train ≈ 42 GB of parts**
  (plus the unified copies during extraction). Set `DELETE_ZIPS_AFTER = True` to reclaim space.

## File naming
`[ID]_[page#]_r[region#]l[line#].[png|txt]` — region/line come from Transkribus layout
recognition. The `[ID]` is the letter ID; look it up at `www.bullinger-digital.ch/letter/[ID]`.

Downloads go to **`data/newbulliger/`** (the repo's `data/` is gitignored). Uses `wget -c`
(resumable) and the `zip` CLI for unifying split sets — make sure both are installed.

In [5]:
import os, zipfile
import pandas as pd
from glob import glob

# Config

In [6]:
# Git-LFS zip archives served via the GitHub media endpoint.
BULLINGER_ROOT = "https://media.githubusercontent.com/media/pstroe/bullinger-htr/main"

# Base archive names per split (by document-ID range), WITHOUT extension.
# train sets are split multi-volume zips (.z01.. + .zip); test sets are single .zip files.
# NB: train has NO 10001-12000 set; that range only exists in test/val.
SPLIT_ARCHIVES = {
    "train": ["0-2000-out", "2001-4000-out", "4001-6000-out",
              "6001-8000-out", "8001-10000-out", "12001-14000-out"],
    "test":  ["0-2000-out", "2001-4000-out", "4001-6000-out", "6001-8000-out",
              "8001-10000-out", "10001-12000-out", "12001-14000-out"],
}

# Local download on this computer, under the gitignored data/ folder.
# Notebook runs from notebooks/, so data/newbulliger is one level up.
DL_DIR      = "../data/newbulliger/zips"   # downloaded .zip volumes + unified zips
EXTRACT_DIR = "../data/newbulliger"        # extracted de/ line crops + CSVs
LANG        = "de"     # "de" (ENHG), "la" (Latin), or None to keep both languages
DELETE_ZIPS_AFTER = False   # True -> remove zip volumes after extraction (reclaims ~40 GB)

# Download + extract

In [7]:
import subprocess, urllib.request

def _remote_exists(url):
    """True if the URL responds 200 to a HEAD request (used to probe split parts)."""
    try:
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD")) as r:
            return r.status == 200
    except Exception:
        return False

def fetch_archive(split, base):
    """Download one archive set (single OR split) into DL_DIR/<split>.
    Returns the path to a directly-openable .zip (unifying split volumes if needed)."""
    dl = os.path.join(DL_DIR, split)
    os.makedirs(dl, exist_ok=True)
    root = f"{BULLINGER_ROOT}/{split}/{base}"

    # Discover split volume parts (.z01, .z02, ...) until one is missing.
    parts, i = [], 1
    while _remote_exists(f"{root}.z{i:02d}"):
        parts.append(f"{base}.z{i:02d}")
        i += 1
    parts.append(f"{base}.zip")            # the final volume is always present

    for p in parts:
        dest = os.path.join(dl, p)
        print(f"  downloading {p} ...")
        !wget -c -q --show-progress -O "{dest}" "{BULLINGER_ROOT}/{split}/{p}"

    final_zip = os.path.join(dl, f"{base}.zip")
    if len(parts) == 1:
        return final_zip                   # single-file archive (test/)

    # Split set -> unify all volumes into one zip with `zip -F` (per the dataset README).
    single = os.path.join(dl, f"single-{base}.zip")
    if not os.path.exists(single):
        print(f"  unifying {len(parts)} split volumes -> single-{base}.zip")
        subprocess.run(["zip", "-F", final_zip, "--out", single],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return single

def get_split(split):
    """Download + extract one split; return a (file_name, text) DataFrame."""
    ext = os.path.join(EXTRACT_DIR, split)
    os.makedirs(ext, exist_ok=True)
    lang_glob = LANG if LANG else "*"

    for base in SPLIT_ARCHIVES[split]:
        print(f"[{split}] {base}")
        zpath = fetch_archive(split, base)
        with zipfile.ZipFile(zpath) as zf:
            members = zf.namelist() if LANG is None else [m for m in zf.namelist() if f"/{LANG}/" in m]
            zf.extractall(ext, members=members)
        print(f"  extracted {len(members)} members from {os.path.basename(zpath)}")
        if DELETE_ZIPS_AFTER:
            for f in glob(os.path.join(DL_DIR, split, f"{base}.z*")) + \
                     glob(os.path.join(DL_DIR, split, f"single-{base}.zip")):
                os.remove(f)

    # Pair each .png line crop with its .txt transcription
    rows = []
    for p in sorted(glob(os.path.join(ext, "**", lang_glob, "*.png"), recursive=True)):
        txt = p[:-4] + ".txt"
        if os.path.exists(txt):
            with open(txt, encoding="utf-8") as f:
                text = f.read().strip()
            if text:
                rows.append({"file_name": p, "text": text})
    return pd.DataFrame(rows).reset_index(drop=True)

In [8]:
train_df = get_split("train")
test_df  = get_split("test")

print(f"\nTrain ({LANG}) lines: {len(train_df)}")
print(f"Test  ({LANG}) lines: {len(test_df)}")
train_df.head()

[train] 0-2000-out
  downloading 0-2000-out.z01 ...
../data/newbulliger 100%[===================>]   2.00G  6.37MB/s    in 5m 26s  
  downloading 0-2000-out.z02 ...
../data/newbulliger 100%[===================>]   2.00G  6.37MB/s    in 5m 23s  
  downloading 0-2000-out.z03 ...
../data/newbulliger 100%[===================>]   2.00G  6.37MB/s    in 5m 24s  
  downloading 0-2000-out.z04 ...
../data/newbulliger 100%[===================>]   2.00G  6.37MB/s    in 5m 26s  
  downloading 0-2000-out.z05 ...
../data/newbulliger 100%[===================>]   2.00G  6.37MB/s    in 5m 24s  
  downloading 0-2000-out.z06 ...
../data/newbulliger 100%[===================>]   2.00G  5.64MB/s    in 5m 35s  
  downloading 0-2000-out.zip ...
  unifying 7 split volumes -> single-0-2000-out.zip
  extracted 11579 members from single-0-2000-out.zip
[train] 2001-4000-out
  downloading 2001-4000-out.z01 ...
../data/newbulliger 100%[===================>]   2.00G  6.37MB/s    in 5m 28s  
  downloading 2001-4000-out

,file_name,text
0,../data/newbulliger/train/0-2000-out/de/1000_0...,teneo.
1,../data/newbulliger/train/0-2000-out/de/1014_0...,mynem gunstigen hern.
2,../data/newbulliger/train/0-2000-out/de/1014_0...,Zurich.
3,../data/newbulliger/train/0-2000-out/de/1019_0...,"es aber publiciert, werden alle engelendische,..."
4,../data/newbulliger/train/0-2000-out/de/1019_0...,oberländisch unnd italianisch kaufleut hinwegk...


# (Optional) Persist the line index as CSV
Saves the file_name/text pairs so other notebooks can load the split without re-extracting.

In [9]:
#train_df.to_csv(os.path.join(EXTRACT_DIR, "train_de.csv"), index=False)
#test_df.to_csv(os.path.join(EXTRACT_DIR, "test_de.csv"), index=False)
#print("Saved train_de.csv and test_de.csv to", EXTRACT_DIR)